In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

%config InlineBackend.figure_format = 'retina'

plt.rcParams['font.family'] = 'Arial'
plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams['font.size'] = 5
plt.rcParams['figure.figsize'] = (3.5, 2.5)
plt.rcParams['figure.dpi'] = 150

plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['xtick.major.size'] = 2
plt.rcParams['ytick.major.size'] = 2


import sys
from pathlib import Path
sys.path.append('../')
from utils import *  # key functions for this project

THRESHOLD = 50


/home/jjusuf/miniconda3/envs/main/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Get search times (CTCF loops)

In [6]:
dt_arr = np.array([1, 2, 5, 10, 30, 60])
loop_num_arr = np.array([0, 1, 2])
noise_arr = np.array([0, 10, 20, 30, 40, 50])

n_reps = 19 - 10 + 1  # 10..19 inclusive
total_iters = len(dt_arr) * len(loop_num_arr) * len(noise_arr) * n_reps

rows_CTCF = []

with tqdm(total=total_iters, desc="calculating search times (initial delay expected)", unit="rep") as pbar:
    for dt in dt_arr:
        for loop_num in loop_num_arr:
            for noise in noise_arr:
                search_times_arr = search_times_all_reps(loop_num, noise, non_sticky=False, threshold=THRESHOLD, target_frame_duration=dt, pbar=pbar, samples_per_event=10)
                for t_search in search_times_arr:
                    rows_CTCF.append(
                        {
                            "loop": loop_num,
                            "noise": noise,
                            "dt": dt,
                            "time": t_search,
                        }
                    )


calculating search times (initial delay expected): 100%|██████████| 1080/1080 [01:41<00:00, 10.60rep/s]


In [7]:
search_times_CTCF = pd.DataFrame(rows_CTCF)
search_times_CTCF.to_csv(f'../data/search_times_CTCF.csv', index=False, float_format="%.2f")


### Get ground-truth CTCF search times

In [3]:
loop_num_arr = np.array([0, 1, 2])

dt = 1
n_reps = 19 - 10 + 1  # 10..19 inclusive
total_iters = len(loop_num_arr) * n_reps

rows_CTCF_actual = []

with tqdm(total=total_iters, desc="calculating search times", unit="rep") as pbar:
    for loop_num in loop_num_arr:
        search_times_arr = search_times_CTCF_all_reps(loop_num, noise=0, non_sticky=False,
                                                        threshold=THRESHOLD, target_frame_duration=dt, pbar=pbar,
                                                        samples_per_event=100)
        for t_search in search_times_arr:
            rows_CTCF_actual.append(
                {
                    "loop": loop_num,
                    "time": t_search,
                }
            )


calculating search times: 100%|██████████| 30/30 [00:03<00:00,  7.78rep/s]


In [4]:
search_times_CTCF_actual = pd.DataFrame(rows_CTCF_actual)
search_times_CTCF_actual.to_csv(f'../data/search_times_CTCF_actual.csv', index=False, float_format="%.2f")


### Get search times (EP loops)

In [ ]:
dt_arr = np.array([0.02, 0.05, 0.1, 0.2, 0.5, 1])
loop_num_arr = np.array([3, 4, 5])
noise_arr = np.array([0, 10, 20, 30, 40, 50])

for dt in dt_arr:
    print(f'working on dt={dt}', flush=True)
    total_iters = len(loop_num_arr) * len(noise_arr)
    with tqdm(total=total_iters, desc="calculating search times", unit="rep") as pbar:
        rows_EP = []
        for loop_num in loop_num_arr:
            for noise in noise_arr:
                search_times_arr = search_times_all_reps(loop_num, noise, non_sticky=False, threshold=THRESHOLD,
                                                         target_frame_duration=dt, pbar=pbar, reps=[10])
                for t_search in search_times_arr:
                    rows_EP.append(
                        {
                            "loop": loop_num,
                            "noise": noise,
                            "time": t_search,
                        }
                    )

    search_times_EP = pd.DataFrame(rows_EP)
    search_times_EP.to_csv(f'../data/search_times_EP_deltaT_{dt}s.csv', index=False, float_format="%.2f")


a
working on dt=0.02


calculating search times:   0%|          | 0/9 [00:25<?, ?rep/s]


KeyboardInterrupt: 